Break your agent on purpose.

Three planted breaks, each restored: a runaway with no stop, a blank
tool description, and a write with no gate then a gate that says no.


In [ ]:
from pathlib import Path
import json
import sys

root = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
sys.path.insert(0, str(root))

from techcorp.agent.scratch import (
    ScratchAgent,
    openai_caller,
    reset_password,
    tool_schema,
)


In [ ]:
# Planted: a tool that always returns "try again" and a loop with no
# real stop. Safety cap of 30 is printed so the process cannot hang.


def try_again() -> str:
    return "try again"


def planted_runaway(_messages: list) -> dict:
    return {
        "content": "",
        "tool_calls": [
            {
                "id": "spin",
                "name": "nudge",
                "arguments": {},
                "arguments_json": "{}",
            }
        ],
    }


SAFETY_CAP = 30
print("safety_cap", SAFETY_CAP)
runaway = ScratchAgent(
    planted_runaway,
    {"nudge": try_again},
    cap=SAFETY_CAP,
)
out_run = runaway.run("keep going")
print("runaway_stop", out_run["stop"])
print("runaway_rounds", len(out_run["rounds"]))
print("runaway_restored", "cap is back")


In [ ]:
# Planted: blank tool description. The model may skip it or misroute.
blank_schema = [
    tool_schema("reset_password", "Reset a TechCorp employee laptop password."),
    {
        "type": "function",
        "function": {
            "name": "wipe_disk",
            "description": "",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {"type": "string"}
                },
                "required": ["user_id"],
            },
        },
    },
]
blank_agent = ScratchAgent(
    openai_caller(blank_schema),
    {
        "reset_password": reset_password,
        "wipe_disk": lambda user_id: json.dumps({"ok": False, "planted": True}),
    },
    cap=8,
)
out_blank = blank_agent.run(
    "TC-1001 user E-4101: please reset my laptop password."
)
blank_names = [row["name"] for row in out_blank["rounds"] if row["kind"] == "tool"]
print("blank_tool", "wipe_disk")
print("blank_choices", blank_names)
print("blank_called_wipe_disk", "wipe_disk" in blank_names)
print("blank_final", out_blank["final"])
print("blank_restored", "description should name the tool")


In [ ]:
# Planted: reset with no gate, then a gate whose input() is replaced by
# a fixed "no" for the capture so the write is refused.
TICKET = "TC-1001 user E-4101: reset my laptop password."
schema = [tool_schema("reset_password", "Reset a TechCorp employee laptop password.")]
caller = openai_caller(schema)

no_gate = ScratchAgent(
    caller,
    {"reset_password": reset_password},
    cap=8,
    gate_writes=False,
)
out_free = no_gate.run(TICKET)
free_names = [row["name"] for row in out_free["rounds"] if row["kind"] == "tool"]
print("no_gate_choices", free_names)
print("no_gate_stop", out_free["stop"])
print("no_gate_final", out_free["final"])

gated = ScratchAgent(
    caller,
    {"reset_password": reset_password},
    cap=8,
    gate_writes=True,
    approve="no",
)
out_gate = gated.run(TICKET)
print("gate_decision", "no")
print("gate_stop", out_gate["stop"])
print("gate_final", out_gate["final"])
print("write_refused", out_gate["stop"] == "gate")
print("gate_restored", "writes park behind a yes")
